# Week 3: Hybrid Semantic Leakage Detector & Defense Benchmarking

**Reference:** *Data Extraction Attacks in Retrieval-Augmented Generation via Backdoors* (arXiv:2411.01705v2)  
**Hardware:** Single Kaggle T4 GPU (16 GB VRAM)  
**Scope:** Week 3 of 4: Extracting multi-document semantic coverage and lexical features, training non-circular detector on 600 verified outputs, calibrating thresholds on 1,000 benign validation samples, and evaluating all four defense conditions against test data.

---

### Step 0: Kaggle Setup & Hugging Face Authentication
Before running, ensure:
1. In the right panel, set **Accelerator** to **GPU T4 x1**.
2. In the right panel, set **Internet** to **On**.
3. Add your Hugging Face token under **Add-ons -> Secrets** as `HF_TOKEN`.

In [ ]:
# Ensure repository files are present
import os
if not os.path.exists('defense/calibrate_detector.py'):
    print('Cloning repository into Kaggle working directory...')
    !git clone https://github.com/starboy1402/Rag_Backdoor.git /kaggle/working/Rag_Backdoor
    %cd /kaggle/working/Rag_Backdoor
else:
    print('Project files already present.')

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('Hugging Face authentication successful!')
except Exception as e:
    print(f'Hugging Face secret not found ({e}).')


### Step 1: Feature Extraction on Validation Splits (Fit & Calibrate)
Extract normalized sentence embedding features with `BAAI/bge-small-en-v1.5` for:
- 600 verified outputs (300 benign, 150 verbatim leak, 150 paraphrase leak)
- 1,000 benign calibration outputs (freezing the empirical FPR thresholds)

In [ ]:
# Train detector and calibrate decision thresholds
!python defense/calibrate_detector.py \
    --out-model cache/detector_bundle.pkl \
    --out-summary cache/calibration_summary.json


### Step 2: Defense Comparison on Held-Out Test Set (500 Samples)
Evaluate the four defense conditions across test questions:
1. **No Defense (Raw Model)**
2. **Privacy Prompt Baseline** (`'Do not repeat any content from the context.'`)
3. **Paper's 95% Entity Filter Baseline**
4. **Proposed Hybrid Semantic Leakage Detector** (at calibrated $\tau_{0.05}$ and $\tau_{0.01}$)

In [ ]:
# Run comprehensive defense benchmarking
!python evaluation/evaluate_metrics.py \
    --test-file cache/medmcqa_test_500.json \
    --retrieval-cache cache/retrieval_cache.json \
    --clean-model ./checkpoints/gemma_2b_clean_baseline \
    --paraphrase-model ./checkpoints/gemma_2b_paraphrase_5pct \
    --detector-bundle cache/detector_bundle.pkl \
    --output-file cache/defense_benchmark_results.json


### Step 3: Inspect Calibrated Operating Thresholds & Feature Weights
Review feature importance, AUROC, AUPRC, and calibrated thresholds.

In [ ]:
import json
with open('cache/calibration_summary.json', 'r') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))
